In [1]:
import snapatac2 as snap
import numpy as np
import pandas as pd
# Set the environment variable to the desired cache path
import os
import scanpy as sc

In [7]:
import pandas as pd
import scanpy as sc
import snapatac2 as snap
import os

# 1. Paths and load
h5ad_path = "/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Snapatac2/IPC/peak_mat.h5ad"
csv_path = "/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scanpy/IPC/ipc_four_lineage.csv"
base_out_dir = "/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/IPC/bigwig"

print("Loading data...")
ipc_peaks = sc.read_h5ad(h5ad_path)
anno = pd.read_csv(csv_path, index_col=0)

# Align anno with ipc_peaks.obs_names
# Expects lineage columns in anno
ipc_peaks.obs = anno.loc[ipc_peaks.obs_names].copy()

# celltype as str/category
ipc_peaks.obs['celltype'] = ipc_peaks.obs['celltype'].astype(str)

# 3. Lineage columns -> output folder names
# Map: "CSV column name" -> output folder name
lineage_map = {
    'is_ExN_lineage': 'ExN',
    'is_InN_lineage': 'InN',
    'is_Oligo_lineage': 'Oligo',
    'is_Astro_lineage': 'Astro'
}

# 4. Loop lineages
for col_name, lineage_name in lineage_map.items():
    print(f"Processing lineage: {lineage_name} (Column: {col_name})...")
    
    # A. Subset cells in lineage
    # 1 = in lineage, 0 = out
    mask = ipc_peaks.obs[col_name] == 1
    
    # Skip empty subsets
    if mask.sum() == 0:
        print(f"  No cells found for {lineage_name}, skipping.")
        continue
        
    # Lineage subset view
    subset_adata = ipc_peaks[mask].copy()
    
    print(f"  Selected {subset_adata.n_obs} cells.")
    print(f"  Cell types in this lineage: {subset_adata.obs['celltype'].unique()}")

    # B. Output dir per lineage
    out_dir = os.path.join(base_out_dir, lineage_name)
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    
    # C. Export BigWig
    # groupby clean celltype
    # Folder name encodes lineage; no extra suffix
    try:
        snap.ex.export_coverage(
            subset_adata, 
            groupby="celltype", 
            out_dir=out_dir, 
            output_format="bigwig",
            suffix=".bw" # optional explicit suffix
        )
        print(f"  Export finished for {lineage_name} -> {out_dir}")
    except Exception as e:
        print(f"  Error exporting {lineage_name}: {e}")

print("All Done.")

Loading data...
Processing lineage: ExN (Column: is_ExN_lineage)...


2026-03-13 17:16:14 - INFO - Exporting fragments...


  Selected 2916 cells.
  Cell types in this lineage: ['NPC' 'IPC' 'ExN_naive']


2026-03-13 17:20:16 - INFO - Creating coverage files...


  Export finished for ExN -> /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/IPC/bigwig/ExN
Processing lineage: InN (Column: is_InN_lineage)...


...

  Export finished for Oligo -> /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/IPC/bigwig/Oligo
Processing lineage: Astro (Column: is_Astro_lineage)...


2026-03-13 17:33:18 - INFO - Exporting fragments...


  Selected 2754 cells.
  Cell types in this lineage: ['APC' 'NPC' 'IPC']


2026-03-13 17:37:04 - INFO - Creating coverage files...


  Export finished for Astro -> /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/IPC/bigwig/Astro
All Done.


In [8]:
peak_df = ipc_peaks.var.index.to_series().str.extract(r'(chr[\w]+):(\d+)-(\d+)')
peak_df.columns = ['chrom', 'start', 'end']
peak_df.to_csv("/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/IPC/outs/consensus_peaks.bed", sep="\t", header=False, index=False)

In [9]:
print(ipc_peaks.var.head())

Empty DataFrame
Columns: []
Index: [chr1:817113-817614, chr1:817788-818289, chr1:827302-827803, chr1:858660-859161, chr1:869693-870194]
